# Airbus Ship Detection — Traditional ML + CNN

This notebook is adapted from the mini-project's `train_models.py` so it can run directly in **Kaggle Notebooks**.

It:
- Finds the original Airbus competition CSV and images attached through **Add Input**.
- Converts the run-length-encoded annotations into image-level labels: **Ship** / **No Ship**.
- Trains five traditional classifiers (Random Forest, Linear SVM, KNN, Gaussian Naive Bayes, LDA) on HSV-histogram + HOG features.
- Trains a CNN and reports accuracy, precision, recall, F1-score, and confusion matrices.
- Saves models and results under `/kaggle/working/airbus_project_outputs/`.

**Important:** This is image-level classification, not ship localization/segmentation. For practicality, traditional ML defaults to a subset. CNN also defaults to a subset; you can switch it to the full dataset in the configuration cell, but full training may take a long time and requires enough Kaggle session time/GPU quota.

## Before running
1. Create a Kaggle Notebook.
2. Click **Add Input** and attach the original **Airbus Ship Detection Challenge** competition data.
3. If Kaggle asks, accept the competition rules.
4. For faster CNN training, set **Settings → Accelerator → GPU** (if available).
5. Run the cells from top to bottom.

No local Windows setup or 31 GB download is required on your PC.

## 1. Configuration
Run the notebook as-is for a manageable first run. To attempt CNN training on every labeled image, change `CNN_LIMIT = 5000` to `CNN_LIMIT = None`. Traditional feature extraction on all images is intentionally avoided because HOG features across the full dataset can be very slow and memory-intensive.

In [ ]:
# Adjust these settings before running the training cells.
IMG_SIZE = 128
SEED = 42

# Traditional ML feature extraction is expensive; start with 3,000 images.
TRADITIONAL_LIMIT = 3000

# CNN starter run. Set to None to stream the entire available dataset in batches.
CNN_LIMIT = 5000

EPOCHS = 5
BATCH_SIZE = 32

OUTPUT_DIR = "/kaggle/working/airbus_project_outputs"
MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

print("Traditional ML image limit:", TRADITIONAL_LIMIT)
print("CNN image limit:", "ALL available images" if CNN_LIMIT is None else CNN_LIMIT)
print("Outputs will be saved to:", OUTPUT_DIR)

## 2. Imports and Kaggle input discovery

In [ ]:
import os
import json
import random
import zipfile
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

random.seed(SEED)
np.random.seed(SEED)

try:
    import tensorflow as tf
    print("TensorFlow:", tf.__version__)
    print("GPU devices:", tf.config.list_physical_devices("GPU"))
except ImportError:
    tf = None
    print("TensorFlow is not installed in this runtime.")

INPUT_ROOT = "/kaggle/input"
if not os.path.isdir(INPUT_ROOT):
    raise FileNotFoundError("This notebook expects a Kaggle Notebook runtime with /kaggle/input.")

csv_candidates = []
image_dirs = []
zip_candidates = []

for root, dirs, files in os.walk(INPUT_ROOT):
    for name in files:
        if name == "train_ship_segmentations_v2.csv":
            csv_candidates.append(os.path.join(root, name))
        elif name.lower() == "train_v2.zip":
            zip_candidates.append(os.path.join(root, name))
    for name in dirs:
        if name == "train_v2":
            image_dirs.append(os.path.join(root, name))

print("CSV candidates:", csv_candidates[:5])
print("Image directories:", image_dirs[:5])
print("Image ZIP candidates:", zip_candidates[:5])

if not csv_candidates:
    raise FileNotFoundError(
        "Could not find train_ship_segmentations_v2.csv. Click Add Input and attach the original Airbus competition data."
    )

CSV_PATH = csv_candidates[0]
IMAGE_DIR = image_dirs[0] if image_dirs else None
IMAGE_ZIP_PATH = zip_candidates[0] if zip_candidates else None

if IMAGE_DIR is None and IMAGE_ZIP_PATH is None:
    raise FileNotFoundError(
        "Found the labels CSV but not train_v2/ or train_v2.zip. Check that the competition image archive is attached."
    )

print("\nUsing CSV:", CSV_PATH)
print("Using image directory:", IMAGE_DIR)
print("Using image ZIP:", IMAGE_ZIP_PATH)

## 3. Read labels and create image-level Ship / No Ship targets

In [ ]:
df = pd.read_csv(CSV_PATH)
required = {"ImageId", "EncodedPixels"}
if not required.issubset(df.columns):
    raise ValueError("CSV must contain ImageId and EncodedPixels columns.")

encoded = df["EncodedPixels"].fillna("").astype(str).str.strip()
df["has_ship"] = (encoded != "") & (encoded != "-1") & (encoded.str.lower() != "nan")
labels = df.groupby("ImageId", as_index=False)["has_ship"].any()
labels["has_ship"] = labels["has_ship"].astype("int32")

print("CSV annotation rows:", len(df))
print("Unique images in labels:", len(labels))
print("Image-level class counts:")
print(labels["has_ship"].value_counts().rename(index={0: "No Ship", 1: "Ship"}))

labels.head()

## 4. Image reader and a few sample images
The image reader works whether Kaggle exposes `train_v2/` as a directory or provides `train_v2.zip`. When the images are in a ZIP, it reads individual files directly instead of extracting the whole archive.

In [ ]:
_zip_handle = zipfile.ZipFile(IMAGE_ZIP_PATH, "r") if IMAGE_ZIP_PATH else None
_zip_names = set(_zip_handle.namelist()) if _zip_handle else set()

def read_rgb_image(image_id, size=IMG_SIZE):
    if IMAGE_DIR is not None:
        path = os.path.join(IMAGE_DIR, image_id)
        if not os.path.isfile(path):
            return None
        image = cv2.imread(path)
    else:
        # ZIPs may contain a top-level train_v2/ directory or just image filenames.
        candidates = [image_id, "train_v2/" + image_id]
        zip_member = next((name for name in candidates if name in _zip_names), None)
        if zip_member is None:
            return None
        try:
            raw = _zip_handle.read(zip_member)
            image = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)
        except KeyError:
            return None

    if image is None:
        return None
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return cv2.resize(image, (size, size), interpolation=cv2.INTER_AREA)

# Keep only rows whose image exists. For ZIPs, use its file listing to avoid reading every image.
if IMAGE_DIR is not None:
    existing = set(os.listdir(IMAGE_DIR))
    labels = labels[labels["ImageId"].isin(existing)].reset_index(drop=True)
else:
    available_members = set()
    for member in _zip_names:
        base = os.path.basename(member)
        if base.lower().endswith((".jpg", ".jpeg", ".png")):
            available_members.add(base)
    labels = labels[labels["ImageId"].isin(available_members)].reset_index(drop=True)

if len(labels) == 0:
    raise RuntimeError("No labeled images matched the image directory/archive. Inspect the paths printed above.")

print("Images available:", len(labels))
print(labels["has_ship"].value_counts().rename(index={0: "No Ship", 1: "Ship"}))

sample_rows = labels.sample(min(8, len(labels)), random_state=SEED).reset_index(drop=True)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax in axes.ravel():
    ax.axis("off")
for ax, row in zip(axes.ravel(), sample_rows.to_dict("records")):
    img = read_rgb_image(row["ImageId"])
    if img is not None:
        ax.imshow(img)
        ax.set_title("Ship" if row["has_ship"] else "No Ship")
plt.tight_layout()
plt.show()

## 5. Traditional ML: extract HSV histogram + HOG features
The traditional models run on a stratified subset (default 3,000 images). This is deliberate: extracting handcrafted features for every image in the full competition dataset can be slow.

In [ ]:
def extract_features(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 8],
                        [0, 180, 0, 256, 0, 256])
    hist = cv2.normalize(hist, hist).flatten()
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    hog_features = hog(
        gray, orientations=9, pixels_per_cell=(16, 16),
        cells_per_block=(2, 2), block_norm="L2-Hys",
        feature_vector=True
    )
    return np.concatenate([hist, hog_features]).astype(np.float32)

def select_stratified_rows(frame, limit, seed=SEED):
    if limit is None or limit <= 0 or len(frame) <= limit:
        return frame.sample(frac=1, random_state=seed).reset_index(drop=True)
    selected, _ = train_test_split(
        frame, train_size=limit, random_state=seed, stratify=frame["has_ship"]
    )
    return selected.reset_index(drop=True)

traditional_rows = select_stratified_rows(labels, TRADITIONAL_LIMIT)
traditional_images = []
traditional_y = []
traditional_names = []

for i, row in traditional_rows.iterrows():
    img = read_rgb_image(row["ImageId"])
    if img is not None:
        traditional_images.append(img)
        traditional_y.append(int(row["has_ship"]))
        traditional_names.append(row["ImageId"])
    if (i + 1) % 250 == 0:
        print(f"Read {i + 1}/{len(traditional_rows)} traditional-ML images")

X_trad_images = np.asarray(traditional_images, dtype=np.uint8)
y_trad = np.asarray(traditional_y, dtype=np.int32)

if len(y_trad) == 0 or len(np.unique(y_trad)) < 2:
    raise ValueError("Traditional ML sample has fewer than two classes. Increase TRADITIONAL_LIMIT.")

trad_train_idx, trad_test_idx = train_test_split(
    np.arange(len(y_trad)), test_size=0.2, random_state=SEED, stratify=y_trad
)

print("Images successfully loaded:", len(y_trad))
print("Ship images:", int((y_trad == 1).sum()))
print("No-ship images:", int((y_trad == 0).sum()))

X_features = []
for i, img in enumerate(X_trad_images):
    X_features.append(extract_features(img))
    if (i + 1) % 250 == 0:
        print(f"Extracted features for {i + 1}/{len(X_trad_images)} images")

X_features = np.asarray(X_features, dtype=np.float32)
X_train, X_test = X_features[trad_train_idx], X_features[trad_test_idx]
y_train, y_test = y_trad[trad_train_idx], y_trad[trad_test_idx]

print("Feature matrix:", X_features.shape)

## 6. Train and evaluate the five traditional classifiers

In [ ]:
def metric_dict(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_score": float(f1_score(y_true, y_pred, zero_division=0))
    }

traditional_models = {
    "random_forest": RandomForestClassifier(
        n_estimators=150, class_weight="balanced", random_state=SEED, n_jobs=-1
    ),
    "svm": make_pipeline(
        StandardScaler(), LinearSVC(class_weight="balanced", random_state=SEED, max_iter=5000)
    ),
    "knn": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "naive_bayes": GaussianNB(),
    "lda": make_pipeline(
        StandardScaler(),
        LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    )
}

results = {
    "dataset": {
        "available_images": int(len(labels)),
        "traditional_images_loaded": int(len(y_trad)),
        "traditional_train_images": int(len(trad_train_idx)),
        "traditional_test_images": int(len(trad_test_idx)),
        "traditional_ship_images": int((y_trad == 1).sum()),
        "traditional_no_ship_images": int((y_trad == 0).sum()),
        "image_size": f"{IMG_SIZE}x{IMG_SIZE}"
    },
    "traditional_models": {}
}
fitted_models = {}
predictions_by_model = {}

for name, model in traditional_models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    scores = metric_dict(y_test, pred)
    results["traditional_models"][name] = scores
    fitted_models[name] = model
    predictions_by_model[name] = pred
    print(scores)

if fitted_models:
    joblib.dump(fitted_models, os.path.join(MODEL_DIR, "traditional_models.joblib"))
    if "random_forest" in fitted_models:
        joblib.dump(fitted_models["random_forest"], os.path.join(MODEL_DIR, "ship_random_forest.joblib"))
        fig, ax = plt.subplots(figsize=(5, 4))
        ConfusionMatrixDisplay(
            confusion_matrix(y_test, predictions_by_model["random_forest"]),
            display_labels=["No Ship", "Ship"]
        ).plot(values_format="d", ax=ax, colorbar=False)
        ax.set_title("Random Forest Confusion Matrix")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, "random_forest_confusion_matrix.png"), dpi=160)
        plt.show()
        print(classification_report(
            y_test, predictions_by_model["random_forest"],
            target_names=["No Ship", "Ship"], zero_division=0
        ))

traditional_results_df = pd.DataFrame(results["traditional_models"]).T
display(traditional_results_df)


## 7. CNN training with a streaming image generator

The CNN reads images in batches, which avoids keeping the entire dataset in RAM. With `CNN_LIMIT = 5000`, it trains on a stratified subset. Change `CNN_LIMIT = None` in Section 1 to attempt all available images.

For full-data training, use a GPU if available and consider reducing `EPOCHS` if the session is time-limited.

In [ ]:
if tf is None:
    print("Skipping CNN: TensorFlow is not available in this runtime.")
    results["cnn"] = {"status": "skipped: TensorFlow unavailable"}
else:
    cnn_rows = select_stratified_rows(labels, CNN_LIMIT).reset_index(drop=True)
    cnn_y = cnn_rows["has_ship"].to_numpy(dtype=np.int32)
    cnn_indices = np.arange(len(cnn_rows))

    if len(np.unique(cnn_y)) < 2:
        raise ValueError("CNN data contains only one class. Increase CNN_LIMIT.")

    cnn_train_idx, cnn_test_idx = train_test_split(
        cnn_indices, test_size=0.2, random_state=SEED, stratify=cnn_y
    )
    cnn_train_idx, cnn_val_idx = train_test_split(
        cnn_train_idx, test_size=0.15, random_state=SEED, stratify=cnn_y[cnn_train_idx]
    )

    class AirbusSequence(tf.keras.utils.Sequence):
        def __init__(self, frame, indices, batch_size=BATCH_SIZE, shuffle=False):
            super().__init__()
            self.frame = frame.reset_index(drop=True)
            self.indices = np.asarray(indices)
            self.batch_size = batch_size
            self.shuffle = shuffle
            self.order = np.arange(len(self.indices))
            self.on_epoch_end()

        def __len__(self):
            return int(np.ceil(len(self.indices) / self.batch_size))

        def __getitem__(self, batch_index):
            batch_positions = self.order[batch_index * self.batch_size:(batch_index + 1) * self.batch_size]
            batch_rows = self.indices[batch_positions]
            batch_x = []
            batch_y = []
            for row_index in batch_rows:
                row = self.frame.iloc[int(row_index)]
                img = read_rgb_image(row["ImageId"])
                if img is None:
                    continue
                batch_x.append(img)
                batch_y.append(int(row["has_ship"]))
            if len(batch_x) == 0:
                return np.zeros((1, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8), np.zeros((1,), dtype=np.float32)
            return np.asarray(batch_x, dtype=np.uint8), np.asarray(batch_y, dtype=np.float32)

        def on_epoch_end(self):
            if self.shuffle:
                np.random.shuffle(self.order)

    train_seq = AirbusSequence(cnn_rows, cnn_train_idx, shuffle=True)
    val_seq = AirbusSequence(cnn_rows, cnn_val_idx, shuffle=False)
    test_seq = AirbusSequence(cnn_rows, cnn_test_idx, shuffle=False)

    cnn = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        tf.keras.layers.Rescaling(1.0 / 255),
        tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    cnn.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall")
        ]
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True
        )
    ]

    history = cnn.fit(
        train_seq,
        validation_data=val_seq,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    cnn.save(os.path.join(MODEL_DIR, "ship_cnn.keras"))

    test_probabilities = cnn.predict(test_seq, verbose=1).ravel()
    cnn_pred = (test_probabilities >= 0.5).astype(np.int32)

    # Sequence can skip unreadable images; rebuild matching labels for a safe metric calculation.
    y_cnn_test = []
    for row_index in cnn_test_idx:
        row = cnn_rows.iloc[int(row_index)]
        if read_rgb_image(row["ImageId"]) is not None:
            y_cnn_test.append(int(row["has_ship"]))
    y_cnn_test = np.asarray(y_cnn_test, dtype=np.int32)

    if len(y_cnn_test) == len(cnn_pred):
        results["cnn"] = metric_dict(y_cnn_test, cnn_pred)
        fig, ax = plt.subplots(figsize=(5, 4))
        ConfusionMatrixDisplay(
            confusion_matrix(y_cnn_test, cnn_pred),
            display_labels=["No Ship", "Ship"]
        ).plot(values_format="d", ax=ax, colorbar=False)
        ax.set_title("CNN Confusion Matrix")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, "cnn_confusion_matrix.png"), dpi=160)
        plt.show()
        print("CNN metrics:", results["cnn"])
    else:
        results["cnn"] = {"error": "Prediction/label count mismatch; inspect unreadable images."}
        print(results["cnn"])

    history_df = pd.DataFrame(history.history)
    display(history_df)
    ax = history_df[["accuracy", "val_accuracy"]].plot(figsize=(8, 4), title="CNN Accuracy")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "cnn_accuracy.png"), dpi=160)
    plt.show()


## 8. Compare results and save metrics

In [ ]:
metrics_path = os.path.join(OUTPUT_DIR, "metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

comparison_rows = []
for model_name, values in results.get("traditional_models", {}).items():
    if "error" not in values:
        comparison_rows.append({"model": model_name, **values})
cnn_values = results.get("cnn", {})
if isinstance(cnn_values, dict) and all(k in cnn_values for k in ["accuracy", "precision", "recall", "f1_score"]):
    comparison_rows.append({"model": "cnn", **cnn_values})

comparison_df = pd.DataFrame(comparison_rows)
if not comparison_df.empty:
    display(comparison_df.sort_values("f1_score", ascending=False).reset_index(drop=True))
    comparison_df.to_csv(os.path.join(OUTPUT_DIR, "model_comparison.csv"), index=False)
else:
    print("No completed model metrics to compare yet.")

print("Metrics saved:", metrics_path)
print("Output contents:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    for name in files:
        print(os.path.join(root, name))

## 9. Package the outputs
At the end of a successful run, create a ZIP containing trained models, metrics, and plots. In Kaggle, files under `/kaggle/working` appear in the notebook's output area and can be downloaded when the session is saved.

In [ ]:
import shutil

zip_base = "/kaggle/working/airbus_ship_detection_results"
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print("Created:", zip_path)
print("If the notebook session is saved, download this ZIP from the notebook output files.")

## Notes for your report
- Report only metrics produced by your actual notebook run; do not use assumed/example accuracy values.
- The dataset is imbalanced, so include precision, recall, F1-score and confusion matrices, not accuracy alone.
- Traditional ML is trained on a subset by design; mention the subset size in your methodology.
- If you set `CNN_LIMIT = None`, mention that the CNN was trained using all images that were accessible to the notebook.
- This model predicts whether an image contains at least one ship. It does not output ship bounding boxes or segmentation masks.